# Week 5 — LoRA Adapter Training (Colab) — traceable record

Trains a precision-targeted **QLoRA** adapter on `qwen2.5:3b` for the ClimateClaimVerifier
claim/opinion classifier, evaluates it on the 100-row eval set, and exports it to **GGUF**
for the Streamlit "Base vs Adapter" demo.

**Verdict (recorded in `project_description.md`):** the adapter shifts the precision/recall
frontier but **no** training balance reaches recall ≥ 0.90 *and* precision ≥ 0.85 together,
so production ships the **base** classifier (recall-first) and this adapter is **demo-only**.
The **40% claim** balance used below is the compromise kept for the demo — see
*Experimentation* at the end for the full ratio sweep that led to it.

Run top-to-bottom for the final 40% adapter. Requires a GPU runtime (Runtime → Change runtime type → T4).

## 1 · Setup

In [1]:
!nvidia-smi -L

GPU 0: Tesla T4 (UUID: GPU-8ea3cc4d-9ad7-6f71-ac71-585d59fcf1b5)


In [1]:
!git clone https://github.com/Sadaf-Burhan/ClimateClaimVerifier.git
%cd ClimateClaimVerifier
!git pull              # latest scripts (build_lora_trainset.py with caching, lora_seed.jsonl)
!mkdir -p tmp/colab

Cloning into 'ClimateClaimVerifier'...
remote: Enumerating objects: 87, done.
remote: Counting objects: 100% (87/87), done.
remote: Compressing objects: 100% (62/62), done.
remote: Total 87 (delta 28), reused 78 (delta 19), pack-reused 0 (from 0)
Receiving objects: 100% (87/87), 188.91 KiB | 10.50 MiB/s, done.
Resolving deltas: 100% (28/28), done.
/content/ClimateClaimVerifier
Already up to date.


In [2]:
# Upload tmp/colab/train_candidates.jsonl (produced locally by `build_lora_trainset.py select`)
from google.colab import files
import shutil
files.upload()
shutil.move("train_candidates.jsonl", "tmp/colab/train_candidates.jsonl")
!wc -l tmp/colab/train_candidates.jsonl     # expect 98

Saving train_candidates.jsonl to train_candidates.jsonl
98 tmp/colab/train_candidates.jsonl


In [3]:
import subprocess, time

# Install zstd, which is required for ollama installation
!sudo apt-get install -y zstd

# Install ollama
!curl -fsSL https://ollama.com/install.sh | sh

# Start ollama serve in a subprocess and wait for it to initialize
subprocess.Popen(["ollama", "serve"]); time.sleep(5)

# Pull the teacher model and install python packages
!ollama pull qwen2.5:7b
!pip -q install ollama pyyaml

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 53 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 0s (2,255 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Selecting previously unselected package zstd.
(Reading database ... 122403 files and directories currently

## 2 · Data generation — teacher labels + manual corrections → master

The hand-labelled seed (`data/lora_seed.jsonl`) covers the hard conspiracy boundary; the
teacher labels the clear bulk. We then apply the Q5 manual corrections (news-headlines the
teacher demoted to opinion) and freeze a **master** so ratio tuning never re-calls the teacher.

In [4]:
!python scripts/build_lora_trainset.py label --teacher qwen2.5:7b
import json
rows = [json.loads(l) for l in open("tmp/colab/lora_trainset.jsonl")]
# Q5 correction: these news-headline-with-commentary posts contain a checkable named
# event/source — flip teacher's 'opinion' back to 'claim' (protects recall).
for r in rows:
    if any(t in r["text"] for t in ["Suffolk Reform council", "Trump uses wartime powers", "Local researchers test a new method"]):
        r["has_claim"] = True; r["reason"] = "Specific named event/source — checkable"
with open("tmp/colab/lora_full.jsonl", "w") as f:        # MASTER — never overwritten
    for r in rows: f.write(json.dumps(r, ensure_ascii=False) + "\n")
print(sum(r["has_claim"] for r in rows), "claim /", sum(not r["has_claim"] for r in rows), "opinion -> master saved")

Cache: 0 already labeled; 98 of 98 candidates left to label.
  labeled 10/98 new
  labeled 20/98 new
  labeled 30/98 new
  labeled 40/98 new
  labeled 50/98 new
  labeled 60/98 new
  labeled 70/98 new
  labeled 80/98 new
  labeled 90/98 new

Wrote /content/ClimateClaimVerifier/tmp/colab/lora_trainset.jsonl: 124 examples (30 claim / 94 opinion). Seed 26 + teacher 98. Leak-free. Re-runs reuse /content/ClimateClaimVerifier/tmp/colab/labeled_cache.jsonl (delete it to force re-labeling).
34 claim / 90 opinion -> master saved


### Q5 validation — review the labels before training

In [5]:
import json, re
from collections import Counter
rows = [json.loads(l) for l in open("tmp/colab/lora_full.jsonl")]
print("balance:", Counter(r["has_claim"] for r in rows))
signal = re.compile(r"\d|°|noaa|nasa|ipcc|environment canada|\brecord\b|%|\bmm\b|\bcm\b|degrees|category\s*\d", re.I)
print("\n-- OPINION-labelled but has a checkable signal (review: should any be CLAIM?) --")
for i, r in enumerate(rows):
    if not r["has_claim"] and signal.search(r["text"]):
        print(i, "|", r["text"][:80])

balance: Counter({False: 90, True: 34})

-- OPINION-labelled but has a checkable signal (review: should any be CLAIM?) --
32 | Reform run councils do not represent local opinion on climate
buff.ly/bGEe2PN
Th
35 | Vigil 235 on 5.6.26 outside #WestYorkshirePensionFund with Global Justice Bradfo
42 | More from the climate emergency.

Spectators sit in the shade while watching a t
45 | Today on World Environment Day, we celebrate the engineers, researchers and stud
48 | Fri 5 Jun

Yes, it's us! We're still here outside the allegedly pro-environment 
53 | It cools down a little in the UK so #r4today simply forgets about the climate em
59 | *APOCALYPSE EMERGENCY: Love's Wake-Up Call* (2013) by Michael Adzema. 

Chapter 
60 | We're excited to welcome the Honourable Kelly Greene (@kellyrichmondbc.bsky.soci
61 | I’m looking forward to the establishment of a UK climate emergency service to pu
68 | At 17, Greta Thunberg said we don't treat the climate emergency like an emergenc
72 | I wrote this 

In [6]:
import json, random
rows = [json.loads(l) for l in open("tmp/colab/lora_full.jsonl")]
for r in random.sample(rows, 15):
    print(r["has_claim"], "|", r["text"][:65], "->", r["reason"])

False | trumpsauthoritarianassault.blogspot.com/2025/08/arct...  ARCTIC I -> Vague, no named events.
True | Aircraft dumped aluminum particles over Lake Erie in April, a lea -> Specific substance and place — checkable even if false
False | United Nations holds emergency summit on climate change impacts i -> Event and place named
True | Kingston Climate Emergency Day 2,649

Local researchers test a ne -> Specific named event/source — checkable
False | Vigil 235 on 5.6.26 outside #WestYorkshirePensionFund with Global -> Vague, no named event/measurement.
True | unitedforclimate.blogspot.com/2025/01/clim... ‘Climate Change Whe -> XR activities measurable.
False | Its gonna remain a climate emergency with or without policies bei -> General viewpoint.
False | Huddersfield hosts screening of national ‘climate emergency brief -> Event description provided
True | Missed that the People Emergency Briefing to hold a UK wide brief -> Event and signatures count mentioned.
False | The world!  Disar

## 3 · Build training set @ 40% claim (chosen balance)

In [7]:
import json, random
full = [json.loads(l) for l in open("tmp/colab/lora_full.jsonl")]
cl = [r for r in full if r["has_claim"]]; op = [r for r in full if not r["has_claim"]]
random.seed(42); random.shuffle(op)
CLAIM_FRAC = 0.40
train = cl + op[:round(len(cl) * (1 - CLAIM_FRAC) / CLAIM_FRAC)]; random.shuffle(train)
with open("tmp/colab/lora_trainset.jsonl", "w") as f:
    for r in train: f.write(json.dumps(r, ensure_ascii=False) + "\n")
print(len(train), "examples,", sum(r["has_claim"] for r in train), "claim /", sum(not r["has_claim"] for r in train), "opinion")

85 examples, 34 claim / 51 opinion


## 4 · Train — QLoRA (Unsloth)

Uses `trl.SFTConfig` (not `TrainingArguments`) to avoid the SFTConfig pickling error, and a FRESH base each run.

In [8]:
!pip -q install unsloth
from unsloth import FastLanguageModel
from datasets import Dataset
from trl import SFTTrainer, SFTConfig
import json

model, tokenizer = FastLanguageModel.from_pretrained(
    "unsloth/Qwen2.5-3B-Instruct-bnb-4bit", max_seq_length=1024, load_in_4bit=True)
model = FastLanguageModel.get_peft_model(
    model, r=8, lora_alpha=16, lora_dropout=0,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    use_gradient_checkpointing="unsloth", random_state=42)

SYSTEM = ("You are a climate claim detector. Decide whether a post contains a verifiable "
  "factual claim about a climate or extreme weather event. A CLAIM has a specific checkable "
  "assertion — a named event, measurement, official source, date, place, or mechanism — "
  "checkable even if false; a checkable fact embedded in a rant is still a CLAIM. An OPINION "
  "has nothing checkable: feelings, jokes, vague conspiracy accusations, personal experiences, "
  "or general viewpoints. First note in `thought` what is checkable, then decide. JSON only.")

def to_text(r):
    tgt = json.dumps({"thought": r.get("thought",""), "has_claim": r["has_claim"], "reason": r.get("reason","")}, ensure_ascii=False)
    return tokenizer.apply_chat_template(
        [{"role":"system","content":SYSTEM},
         {"role":"user","content":f'Post: "{r["text"][:500]}"'},
         {"role":"assistant","content":tgt}], tokenize=False)

rows = [json.loads(l) for l in open("tmp/colab/lora_trainset.jsonl")]
ds = Dataset.from_dict({"text": [to_text(r) for r in rows]})
SFTTrainer(model=model, tokenizer=tokenizer, train_dataset=ds,
    args=SFTConfig(dataset_text_field="text", max_seq_length=1024,
        per_device_train_batch_size=2, gradient_accumulation_steps=4,
        warmup_steps=5, num_train_epochs=3, learning_rate=2e-4,
        logging_steps=5, optim="adamw_8bit", seed=42, output_dir="outputs",
        save_strategy="no")).train()
model.save_pretrained("climate_claim_lora")
print("adapter trained @ 40% claim and saved -> climate_claim_lora/")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.0/74.0 MB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 37.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.6/915.6 MB 692.9 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 59.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.3/322.3 MB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 127.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 37.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 71.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 122.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3

model.safetensors:   0%|          | 0.00/2.05G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.34k [00:00<?, ?B/s]

...

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 85 | Num Epochs = 3 | Total steps = 33
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 14,966,784 of 3,100,905,472 (0.48% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
5,5.436570
10,3.165045
15,2.109382
20,1.178876
25,1.062177
30,0.910117


adapter trained @ 40% claim and saved -> climate_claim_lora/


## 5 · Evaluate the adapter on the 100-row eval set

Served on the lean prompt it was trained on. Base reference (8-shot Ollama, single-post) ≈ recall 0.938 / precision 0.703. Target: recall ≥ 0.90 AND precision ≥ 0.85; the 50k-acres post must stay `claim`.

In [9]:
import json, re, csv
from unsloth import FastLanguageModel
FastLanguageModel.for_inference(model)
eval_rows = list(csv.DictReader(open("data/claim_eval.csv", encoding="utf-8")))

def classify(text):
    p = tokenizer.apply_chat_template(
        [{"role":"system","content":SYSTEM}, {"role":"user","content":f'Post: "{text[:500]}"'}],
        tokenize=False, add_generation_prompt=True)
    ins = tokenizer(p, return_tensors="pt").to("cuda")
    out = model.generate(**ins, max_new_tokens=120, do_sample=False)
    txt = tokenizer.decode(out[0][ins.input_ids.shape[1]:], skip_special_tokens=True)
    m = re.search(r"\{.*\}", txt, re.DOTALL)
    try: return "claim" if (m and json.loads(m.group()).get("has_claim")) else "opinion"
    except: return "opinion"

pr = [classify(r["post_text"]) for r in eval_rows]
tp = sum(a=="claim" and r["expected_label"]=="claim" for r,a in zip(eval_rows,pr))
fn = sum(a=="opinion" and r["expected_label"]=="claim" for r,a in zip(eval_rows,pr))
fp = sum(a=="claim" and r["expected_label"]=="opinion" for r,a in zip(eval_rows,pr))
print(f"ADAPTER @40%: recall={tp/(tp+fn or 1):.3f}  precision={tp/(tp+fp or 1):.3f}  FN={fn}  FP={fp}")
print("50k test:", classify("50,000 acres of ice sheets have melted, what a lie these governments are trying to convince people of Alaska"))

Both `max_new_tokens` (=120) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=1

ADAPTER @40%: recall=0.854  precision=0.788  FN=7  FP=11
50k test: claim


## 6 · Export to GGUF (for the Streamlit demo)

Unsloth merges the adapter, converts via llama.cpp, and quantizes in one call. Download, then register locally: `ollama create qwen2.5-3b-claim-lora -f models/Modelfile`.

In [10]:
model.save_pretrained_gguf("climate_claim_gguf", tokenizer, quantization_method="q4_k_m")


Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/757 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in climate_claim_gguf/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


...


Unsloth: Preparing safetensor model files: 100%|██████████| 2/2 [04:49<00:00, 144.95s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [01:47<00:00, 53.90s/it]


Unsloth: Merge process complete. Saved to `/content/ClimateClaimVerifier/climate_claim_gguf`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Installing prebuilt llama.cpp b9827-mix-1f1aaa4 (app-b9827-mix-1f1aaa4-linux-x64-cpu.tar.gz) - skipping compilation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['climate_claim_gguf_gguf/Qwen2.5-3B-Instruct.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GG

IndexError: list index out of range

In [11]:
import glob, os
from google.colab import files
g = glob.glob("**/*.gguf", recursive=True)[0]
print("GGUF:", g, f"({os.path.getsize(g)/1e9:.1f} GB)")
files.download(g)

GGUF: climate_claim_gguf_gguf/Qwen2.5-3B-Instruct.Q4_K_M.gguf (1.9 GB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

---
## Experimentation — the ratio sweep that led to 40%

Class balance is a **dial** that trades recall against precision. Holding everything else
constant, three training balances were measured on the same eval:

| training balance | recall | precision | note |
|---|---|---|---|
| ~26% claim (full, opinion-heavy) | 0.812 | 0.951 | overshot → precision, recall fails |
| 50% claim (balanced) | 0.958 | 0.742 | overshot → recall, precision barely > base |
| **40% claim (chosen)** | 0.854 | 0.788 | the compromise kept for the demo |

No balance reaches recall ≥ 0.90 **and** precision ≥ 0.85 together — the measured finding
behind the "ship base, no adapter" verdict.

The cells below reproduce the other two balances. **They overwrite `lora_trainset.jsonl` and,
after a retrain, the 40% adapter** — run only to reproduce the sweep, then re-run sections 3–6
to restore the 40% adapter.

> Historical note: the first training run used `transformers.TrainingArguments`, which raised
> a `PicklingError` on `SFTConfig` at the final save (the adapter weights still wrote to the
> checkpoint). The fix is `trl.SFTConfig` with `save_strategy="no"`, used in section 4.

In [ ]:
# EXPERIMENT — 50/50 balance (then re-run the Train + Eval cells in sections 4-5)
import json, random
full = [json.loads(l) for l in open("tmp/colab/lora_full.jsonl")]
cl = [r for r in full if r["has_claim"]]; op = [r for r in full if not r["has_claim"]]
random.seed(42); random.shuffle(op)
train = cl + op[:len(cl)]; random.shuffle(train)        # 50% claim
with open("tmp/colab/lora_trainset.jsonl", "w") as f:
    for r in train: f.write(json.dumps(r, ensure_ascii=False) + "\n")
print("50/50:", len(train), "examples,", sum(r["has_claim"] for r in train), "claim   -> measured 0.958 / 0.742")

In [ ]:
# EXPERIMENT — full opinion-heavy set, ~26% claim (then re-run Train + Eval)
import json, random
full = [json.loads(l) for l in open("tmp/colab/lora_full.jsonl")]
random.seed(42); random.shuffle(full)
with open("tmp/colab/lora_trainset.jsonl", "w") as f:
    for r in full: f.write(json.dumps(r, ensure_ascii=False) + "\n")
print("~26% claim:", len(full), "examples,", sum(r["has_claim"] for r in full), "claim   -> measured 0.812 / 0.951")